# 🎙️ Podcast Summarizer — Week 3 Task

**Author:** Victor Conqueror

This notebook takes a podcast audio file (MP3), transcribes it using HuggingFace's Whisper model,
and then uses Meta's Llama 3.2 (quantized to 4-bit) to generate a structured podcast summary.

**Everything runs on HuggingFace models only — no OpenAI API needed!**

### What you'll learn:
- How to use HuggingFace `pipeline()` for audio transcription (Whisper)
- How to load a large language model with 4-bit quantization (BitsAndBytes)
- How to use `apply_chat_template()` for structured prompting
- How to stream LLM output token-by-token with `TextStreamer`

### Requirements:
- Run on Google Colab with a **T4 GPU** (free tier works!)
- A HuggingFace account with access to Llama 3.2
- An MP3 podcast file uploaded to Google Drive

## Step 0: Install Dependencies

We need specific versions of `transformers`, `bitsandbytes` (for 4-bit quantization),
and `accelerate` (for smart model loading across devices).

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

## Step 1: Imports

Key HuggingFace imports explained:
- `AutoTokenizer` — converts text ↔ numbers (tokens) so the model can understand it
- `AutoModelForCausalLM` — loads a text-generation model (like Llama)
- `TextStreamer` — prints tokens as they're generated (live streaming!)
- `BitsAndBytesConfig` — configures 4-bit quantization to fit big models in small GPUs
- `pipeline` — a high-level helper that bundles model + tokenizer + preprocessing into one call

In [ ]:
import os
import torch
from IPython.display import Markdown, display
from google.colab import drive, userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
    pipeline,
)

## Step 2: Constants & Setup

We define which models to use:
- **Whisper** (`openai/whisper-medium.en`) — an open-source speech-to-text model hosted on HuggingFace
- **Llama 3.2** (`meta-llama/Llama-3.2-3B-Instruct`) — Meta's 3-billion parameter instruction-tuned model

In [ ]:
# Models
WHISPER_MODEL = "openai/whisper-medium.en"
LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

## Step 3: Connect Google Drive & Load Audio

Mount your Google Drive so the notebook can access your podcast file.

**Instructions:**
1. Create a folder called `llms` in your Google Drive
2. Upload an MP3 podcast file into that folder
3. Update the `audio_filename` below with your file's name

In [ ]:
# Mount Google Drive
drive.mount("/content/drive")

# Point to your podcast file — change the filename to match yours!
audio_filename = "/content/drive/MyDrive/llms/podcast_extract.mp3"

# Verify the file exists
if os.path.exists(audio_filename):
    file_size_mb = os.path.getsize(audio_filename) / (1024 * 1024)
    print(f"✅ Found audio file: {audio_filename}")
    print(f"📁 File size: {file_size_mb:.1f} MB")
else:
    print(f"❌ File not found: {audio_filename}")
    print("Please upload your podcast MP3 to Google Drive in the 'llms' folder.")

## Step 4: Sign in to HuggingFace Hub

Llama 3.2 is a **gated model** — you need to:
1. Go to [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) and request access
2. Create a HuggingFace token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Add it to Colab Secrets (🔑 icon in left sidebar) as `HF_TOKEN`

In [ ]:
# Login to HuggingFace using your token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("✅ Logged in to HuggingFace Hub!")

---

## 🎧 PART 1: Transcribe with Whisper

### What is `pipeline()`?

Think of `pipeline()` as a **one-line shortcut** that:
1. Downloads the model from HuggingFace Hub
2. Downloads the matching tokenizer/processor
3. Sets up all the preprocessing (audio → mel spectrogram → tokens)
4. Runs inference
5. Post-processes the output back to text

Without `pipeline()`, you'd need ~20 lines of code to do all that manually!

In [ ]:
# Create the transcription pipeline
# - model: which Whisper model to use
# - dtype: float16 uses half the memory of float32
# - device: 'cuda' = use the GPU
# - return_timestamps: include timing info in the output

whisper_pipe = pipeline(
    task="automatic-speech-recognition",
    model=WHISPER_MODEL,
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True,
)

print("✅ Whisper pipeline loaded!")

In [ ]:
# Run the transcription — just pass the file path!
print("🎙️ Transcribing podcast... (this may take a minute)\n")

result = whisper_pipe(audio_filename)
transcription = result["text"]

print("=" * 60)
print("📝 TRANSCRIPTION RESULT:")
print("=" * 60)
print(transcription)
print(f"\n📊 Word count: {len(transcription.split())}")

---

## 🧠 PART 2: Summarize with Llama 3.2

Now we take the transcription and feed it to Llama 3.2 to produce a structured summary.

### What is Quantization?

Llama 3.2 3B normally needs ~6 GB of GPU memory (in float16). With **4-bit quantization**,
we compress the model weights from 16 bits → 4 bits per parameter, which:
- **Cuts memory usage by ~4x** (fits on a free Colab T4!)
- **Barely affects quality** thanks to NF4 (Normal Float 4) format

The `BitsAndBytesConfig` controls this compression.

In [ ]:
# Configure 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load weights in 4-bit format
    bnb_4bit_use_double_quant=True,  # Extra compression on the quantization constants
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use bfloat16 for computations (good balance)
    bnb_4bit_quant_type="nf4",      # Normal Float 4 — optimized for neural networks
)

print("✅ Quantization config ready!")

### Loading the Tokenizer & Model

**Tokenizer** = Translator between human text and numbers.
- `"Hello world"` → `[15043, 1917]` (encoding)
- `[15043, 1917]` → `"Hello world"` (decoding)

**Model** = The actual neural network that generates new tokens.
- Takes in a sequence of token IDs
- Predicts the next token, one at a time
- `device_map="auto"` automatically places model layers on GPU/CPU as needed

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)
tokenizer.pad_token = tokenizer.eos_token  # Required for Llama

# Load the model with 4-bit quantization
print("⏳ Loading Llama 3.2 (4-bit quantized)... this takes about 1-2 minutes...")
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    device_map="auto",
    quantization_config=quant_config,
)

print("✅ Model loaded!")
print(f"📊 Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

### Building the Prompt

We use a **chat template** with two roles:
- `system` — tells the model HOW to behave (its identity and rules)
- `user` — provides the actual task (the transcription to summarize)

`apply_chat_template()` converts this into the exact token format Llama expects,
including special tokens like `<|begin_of_text|>`, `<|start_header_id|>`, etc.

In [ ]:
# Define the system prompt — tells Llama what kind of output we want
system_message = """
You are an expert podcast summarizer. Given a transcript of a podcast episode,
produce a well-structured summary in markdown (without code blocks) that includes:

1. **Episode Overview** — A 2-3 sentence high-level summary
2. **Key Topics Discussed** — Bullet points of the main subjects covered
3. **Notable Quotes** — Any memorable or impactful statements (in quotes)
4. **Key Takeaways** — The most important insights a listener should remember
5. **Who Should Listen** — What kind of audience would benefit from this episode

Keep it concise but comprehensive. Use a friendly, engaging tone.
"""

# Build the user prompt with the transcription embedded
user_prompt = f"""
Below is a transcript from a podcast episode.
Please create a structured summary following the format specified.

Transcription:
{transcription}
"""

# Assemble the messages in chat format
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt},
]

print("✅ Prompt ready!")
print(f"📊 System prompt: {len(system_message.split())} words")
print(f"📊 User prompt: {len(user_prompt.split())} words (includes transcription)")

### Generate the Summary (with Streaming!)

**TextStreamer** prints each token as it's generated — so you see the output
appearing word-by-word in real time, just like ChatGPT!

In [ ]:
# Convert messages to token IDs using the chat template
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# Set up real-time streaming
streamer = TextStreamer(tokenizer)

# Generate!
print("🧠 Generating podcast summary...\n")
print("=" * 60)

outputs = model.generate(
    inputs,
    max_new_tokens=2000,  # Maximum length of the summary
    streamer=streamer,    # Stream tokens as they're generated
)

print("=" * 60)
print("\n✅ Summary generation complete!")

### Display the Summary (Rendered Markdown)

In [ ]:
# Decode the full output and display as formatted markdown
full_response = tokenizer.decode(outputs[0])

# Extract only the assistant's response (skip the prompt)
# Llama uses <|start_header_id|>assistant<|end_header_id|> to mark the response
if "assistant" in full_response:
    summary = full_response.split("assistant")[-1]
    # Clean up any remaining special tokens
    summary = summary.replace("<|end_header_id|>", "")
    summary = summary.replace("<|eot_id|>", "")
    summary = summary.replace("<|begin_of_text|>", "")
    summary = summary.strip()
else:
    summary = full_response

display(Markdown("# 🎙️ Podcast Summary\n\n" + summary))

---

## 🔍 PART 3: Compare Transcription vs Summary

Let's see the raw transcription side-by-side with our polished summary.

In [ ]:
display(Markdown("## 📝 Raw Transcription\n\n" + transcription))
print("\n" + "=" * 60 + "\n")
display(Markdown("## 🎯 AI-Generated Summary\n\n" + summary))

---

## 📊 BONUS: Podcast Stats

In [ ]:
transcript_words = len(transcription.split())
summary_words = len(summary.split())
compression = (1 - summary_words / transcript_words) * 100 if transcript_words > 0 else 0

stats = f"""
## 📊 Podcast Stats

| Metric | Value |
|--------|-------|
| Transcription words | {transcript_words:,} |
| Summary words | {summary_words:,} |
| Compression ratio | {compression:.1f}% reduction |
| Whisper model | {WHISPER_MODEL} |
| LLM model | {LLAMA_MODEL} |
| Quantization | 4-bit NF4 |
| Model memory | {model.get_memory_footprint() / 1e9:.2f} GB |
"""

display(Markdown(stats))

---

## 🎓 What You Learned (HuggingFace Concepts)

### The HuggingFace Ecosystem — A Cheat Sheet

| Concept | What it does | Used in this notebook |
|---------|-------------|----------------------|
| **HuggingFace Hub** | A platform hosting 500k+ models — like GitHub but for AI models | Downloading Whisper and Llama |
| **`pipeline()`** | One-line shortcut: downloads model + tokenizer, runs inference | Whisper transcription |
| **`AutoTokenizer`** | Text ↔ token IDs converter (every model has its own!) | Converting prompt to Llama tokens |
| **`AutoModelForCausalLM`** | Loads a text-generation model | Loading Llama 3.2 |
| **`BitsAndBytesConfig`** | 4-bit quantization — shrinks models to fit in small GPUs | Fitting 3B params in a T4 |
| **`TextStreamer`** | Prints tokens as they're generated (live output) | Real-time summary generation |
| **`apply_chat_template()`** | Formats messages into model-specific token format | Building the Llama prompt |
| **`device_map="auto"`** | Automatically splits model across GPU/CPU | Smart memory management |

### The Flow
```
Audio File (MP3)
    ↓ pipeline("automatic-speech-recognition")
Raw Text (Transcription)
    ↓ apply_chat_template() + tokenizer
Token IDs
    ↓ model.generate() with TextStreamer
Structured Summary (Markdown)
```